# Ноутбук со сравнением 3 датасетов

## 1. Импорт бибилиотек и конфигурация проекта

In [1]:
# mlflow ui --backend-store-uri sqlite:///C:/project/car-price-analyzer/src/mlflow_runs/mlflow.db

In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import Ridge
from category_encoders.cat_boost import CatBoostEncoder
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV, KFold, RandomizedSearchCV
import copy
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.model_selection import cross_validate
import xgboost as xgb
import pyarrow
import phik
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
import catboost
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate
)
from catboost import CatBoostRegressor
import mlflow.sklearn
from phik.report import plot_correlation_matrix
import plotly
import mlflow
import time
from scipy.stats import randint, uniform, loguniform
import os
from datetime import datetime
import category_encoders as ce
import joblib

c:\Users\Степан\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Создадим словарь конфигураций.

CONFIG = {
    # Константы
    "DEV_MODE": False,
    "DEV_SAMPLE_SIZE": 100000,
    "RANDOM_STATE": 42,
    # Целевая переменная 
    "TARGET": "Цена",
    "YEAR": datetime.now().year
}

In [4]:
df_raw = pd.read_parquet("../data/raw/df_optimal.parquet")

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    df_raw.drop(columns=[CONFIG["TARGET"]]), df_raw[CONFIG["TARGET"]], 
    test_size=0.2, 
    random_state=CONFIG["RANDOM_STATE"]
)

train_cleaned = pd.read_parquet("../data/cleaned/train_cleaned.parquet")
test_cleaned = pd.read_parquet("../data/cleaned/test_cleaned.parquet")

train_features = pd.read_parquet("../data/features/train_features.parquet")
test_features = pd.read_parquet("../data/features/test_features.parquet")

train_optimized = pd.read_parquet("../data/optimized/train_optimized.parquet")
test_optimized = pd.read_parquet("../data/optimized/test_optimized.parquet")

y_train_cleaned = train_cleaned[CONFIG["TARGET"]]
X_train_cleaned = train_cleaned.drop(columns=[CONFIG["TARGET"]])
y_test_cleaned = test_cleaned[CONFIG["TARGET"]]
X_test_cleaned = test_cleaned.drop(columns=[CONFIG["TARGET"]])

y_train_features = train_features[CONFIG["TARGET"]]
X_train_features = train_features.drop(columns=[CONFIG["TARGET"]])
y_test_features = test_features[CONFIG["TARGET"]]
X_test_features = test_features.drop(columns=[CONFIG["TARGET"]])

X_train_optimized = train_optimized.drop(columns=[CONFIG["TARGET"]])
y_train_optimized = train_optimized[CONFIG["TARGET"]]
X_test_optimized = test_optimized.drop(columns=[CONFIG["TARGET"]])
y_test_optimized = test_optimized[CONFIG["TARGET"]]

dir = 'C:/project/car-price-analyzer/src/mlflow_runs'
os.makedirs(dir, exist_ok=True)
mlflow.set_tracking_uri(f'sqlite:///{dir}/mlflow.db')
mlflow.set_experiment('dataset_compare')

<Experiment: artifact_location='file:c:/project/car-price-analyzer/researches/mlruns/4', creation_time=1785854392363, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1785854392363, lifecycle_stage='active', name='dataset_compare', tags={}, trace_location=None, workspace='default'>

In [5]:
scoring = {
    'mape': 'neg_mean_absolute_percentage_error',
    'mae': 'neg_mean_absolute_error'
}

## 2. Запуск эксперимента

In [6]:
datasets = {
    '1_raw_dataset': (X_train_raw, y_train_raw, X_test_raw, y_test_raw),
    '2_cleaned_dataset': (X_train_cleaned, y_train_cleaned, X_test_cleaned, y_test_cleaned),
    '3_features_dataset': (X_train_features, y_train_features, X_test_features, y_test_features),
    '4_optimized_dataset': (X_train_optimized, y_train_optimized, X_test_optimized, y_test_optimized)
}
cv = KFold(n_splits=5, shuffle=True, random_state=CONFIG["RANDOM_STATE"])
params = {
    'bootstrap_type': 'Bernoulli',
    'iterations': 2500,
    'learning_rate': 0.042248138023520114,
    'depth': 10,
    'l2_leaf_reg': 9.394802632887766,
    'random_strength': 0.8303775333704183,
    'border_count': 207,
    'subsample': 0.5483853226992782,
    'loss_function': 'MAE',
    'allow_writing_files': False,
    'eval_metric': 'MAE',
    'random_seed': 42,
    'thread_count': -1,
    'task_type': 'GPU',
    'verbose': 500
}

In [10]:
for run_name, (X_train, y_train, X_test, y_test) in datasets.items():
    print(f"Запуск эксперимента для: {run_name}...")

    X_train = X_train.copy()
    X_test = X_test.copy()
    y_train = y_train.copy()
    y_test = y_test.copy()

    raw_text_features = ['Комплектация', 'Название машины']
    text_features = [col for col in raw_text_features if col in X_train.columns]

    cat_features = [
        col for col in X_train.select_dtypes(include=['object', 'category']).columns 
        if col not in text_features
    ]

    for col in text_features + cat_features:
        X_train[col] = X_train[col].astype(object).fillna('Unknown').astype(str)
        if col in X_test.columns:
            X_test[col] = X_test[col].astype(object).fillna('Unknown').astype(str)

    fit_params = {'cat_features': cat_features}
    if text_features:
        fit_params['text_features'] = text_features

    model = CatBoostRegressor(**params)
    wrapped_model = TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1
    )

    with mlflow.start_run(run_name=run_name, nested=True):

        scores = cross_validate(
            wrapped_model,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            params=fit_params
        )

        cv_mape = -scores['test_mape'].mean()
        cv_mae = -scores['test_mae'].mean()

        wrapped_model.fit(
            X_train, 
            y_train, 
            **fit_params
        )
        y_pred_test = wrapped_model.predict(X_test)

        test_mape = mean_absolute_percentage_error(y_test, y_pred_test)
        test_mae = mean_absolute_error(y_test, y_pred_test)

        # Логирование
        mlflow.log_metric('cv_mape', cv_mape)
        mlflow.log_metric('cv_mae', cv_mae)
        mlflow.log_metric('test_mape', test_mape)
        mlflow.log_metric('test_mae', test_mae)

        print(f"{run_name}:\n"
              f"CV MAPE: {cv_mape:.4f} | CV MAE: {cv_mae:.0f} руб.\n"
              f"TEST MAPE: {test_mape:.4f} | TEST MAE: {test_mae:.0f} руб.\n")

Запуск эксперимента для: 1_raw_dataset...


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.8018503	total: 494ms	remaining: 20m 34s
500:	learn: 0.1610639	total: 1m	remaining: 4m 1s
1000:	learn: 0.1507856	total: 1m 41s	remaining: 2m 32s
1500:	learn: 0.1448677	total: 2m 30s	remaining: 1m 40s
2000:	learn: 0.1402229	total: 3m 26s	remaining: 51.5s
2499:	learn: 0.1363647	total: 4m 22s	remaining: 0us


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.8014537	total: 426ms	remaining: 17m 45s
500:	learn: 0.1610463	total: 1m	remaining: 4m
1000:	learn: 0.1507832	total: 1m 41s	remaining: 2m 32s
1500:	learn: 0.1450961	total: 2m 31s	remaining: 1m 40s
2000:	learn: 0.1406076	total: 3m 24s	remaining: 51.1s
2499:	learn: 0.1364790	total: 4m 23s	remaining: 0us


...

0:	learn: 0.6897738	total: 322ms	remaining: 13m 24s
500:	learn: 0.1563502	total: 39.2s	remaining: 2m 36s
1000:	learn: 0.1488013	total: 1m 8s	remaining: 1m 42s
1500:	learn: 0.1442988	total: 1m 41s	remaining: 1m 7s
2000:	learn: 0.1413022	total: 2m 13s	remaining: 33.2s
2499:	learn: 0.1379029	total: 2m 50s	remaining: 0us


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.6906795	total: 292ms	remaining: 12m 8s
500:	learn: 0.1564489	total: 39.2s	remaining: 2m 36s
1000:	learn: 0.1489160	total: 1m 8s	remaining: 1m 42s
1500:	learn: 0.1441982	total: 1m 42s	remaining: 1m 8s
2000:	learn: 0.1412445	total: 2m 14s	remaining: 33.4s
2499:	learn: 0.1384164	total: 2m 47s	remaining: 0us


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.6902084	total: 291ms	remaining: 12m 6s
500:	learn: 0.1563041	total: 46.8s	remaining: 3m 6s
1000:	learn: 0.1491160	total: 1m 21s	remaining: 2m 2s
1500:	learn: 0.1456508	total: 1m 56s	remaining: 1m 17s
2000:	learn: 0.1422274	total: 2m 38s	remaining: 39.6s
2499:	learn: 0.1395371	total: 3m 20s	remaining: 0us
4_optimized_dataset:
CV MAPE: 0.1545 | CV MAE: 144371 руб.
TEST MAPE: 0.1529 | TEST MAE: 139173 руб.



### Предсказание для единичиной машины (Kia Rio IV). Проверим, правда ли метрика на сыром датасете такая хорошая

In [ ]:
car_to_predict = {
    "Название машины": "Kia Rio IV Рестайлинг",
    "Год": 2021,
    "Ссылка": "https://auto.ru/cars/used/sale/kia/rio/1120485910-c4d92/",
    "Дата размещения объявления": "2023-11-20",
    "Кол-во просмотров": 1420,
    "Скрыто": np.nan,
    "Объем двигателя": 1.6,
    "Тип двигателя": "Бензин",
    "Мощность": 123.0,
    "Коробка передач": "Автомат",
    "Привод": "Передний",
    "Пробег": 42000.0,
    "Руль": "Левый",
    "Поколение": 4,
    "Рестайлинг": 1,
    "Цвет": "Серый",
    "Комплектация": "Style",
    "Владелец": "Частное лицо",
    "Особые отметки": np.nan,
    "Тип кузова": "Седан",
    "VIN": np.nan,
    "Проверено": np.nan,
    "Номер кузова": np.nan,
    "Метка": "Kia",
    "Город": "Москва",
    "Регион": "Московская область",
    "Макро-регион": "Центральный ФО",
    "Рефрижератор": np.nan,
    "Спальник": np.nan,
    "Объем кузова": np.nan,
    "Категория ТС": np.nan,
    "Тип техники": np.nan,
    "Колёсная формула": np.nan,
    "Количество осей": np.nan,
    "Масса": np.nan,
    "Количество мест": np.nan,
    "Грузоподъемность": np.nan,
    "Длина кузова": np.nan,
    "Топливо": np.nan,
    "Тип кабины": np.nan,
    "Ходовая часть": np.nan,
    "Рабочая ширина": np.nan,
    "Моточасы": np.nan,
    "Оборудование": np.nan,
    "Владельцы": 1,
    "Высота седла": np.nan,
    "Бренд ХОУ": np.nan,
    "Объем ковша": np.nan,
    "Длина стрелы": np.nan,
    "Грузоподъемность стрелы": np.nan,
    "Высота вышки": np.nan,
    "Состояние": np.nan,
    "Страна производства": np.nan,
    "Высота подъема": np.nan,
    "Ошибка_ст": np.nan,
    "Ошибка_знач": np.nan,
    "Пропуски в данных": np.nan,
    "Марка": "Kia",
    "Модель": "Rio",
    "Есть особые отметки": 0,
    "Возраст": 2,
    "Пробег за год": 21000.0,
    "Литровая мощность": 76.875,
    "Подозрительный пробег": 0,
    "Тип продавца": "Частное лицо",
    "Класс бренда": "Масс-маркет",
    "Месяц_размещения": 11,
    "Год_размещения": 2023
}

X_single_raw = pd.DataFrame([car_to_predict])

mlflow.set_experiment('single_car_compare')

first_dataset, *_, last_dataset = datasets.items()

for run_name, (X_train, y_train, X_test, y_test) in (first_dataset, last_dataset):
    print(f"\nЗапуск эксперимента единичной машины для: {run_name}...")

    X_train = X_train.copy()
    y_train = y_train.copy()

    X_single = X_single_raw[X_train.columns].copy()

    raw_text_features = ['Комплектация', 'Название машины']
    text_features = [col for col in raw_text_features if col in X_train.columns]

    cat_features = [
        col for col in X_train.select_dtypes(include=['object', 'category']).columns 
        if col not in text_features
    ]

    for col in text_features + cat_features:
        X_train[col] = X_train[col].astype(object).fillna('Unknown').astype(str)
        if col in X_single.columns:
            X_single[col] = X_single[col].astype(object).fillna('Unknown').astype(str)

    num_features = [col for col in X_train.columns if col not in text_features + cat_features]
    for col in num_features:
        if col in X_single.columns:
            X_single[col] = pd.to_numeric(X_single[col], errors='coerce')

    fit_params = {'cat_features': cat_features}
    if text_features:
        fit_params['text_features'] = text_features

    model = CatBoostRegressor(**params) 
    wrapped_model = TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1
    )

    with mlflow.start_run(run_name=run_name):
        wrapped_model.fit(
            X_train, 
            y_train, 
            **fit_params
        )

        single_pred = wrapped_model.predict(X_single)[0]
        actual_price = 1740000.0
        error = abs(single_pred - actual_price)
        error_percentage = error / actual_price

        mlflow.log_metric('single_prediction', float(single_pred))
        mlflow.log_metric('actual_price', float(actual_price))
        mlflow.log_metric('error', float(error))
        mlflow.log_metric('error_percentage', float(error_percentage))

        print(f"Предсказание для машины (Kia Rio 2021): {single_pred:,.0f} руб.")
        print(f"Реальная цена из объявления: {actual_price:,.0f} руб.")
        print(f"Ошибка предсказания: {error:,.0f} руб. ({error_percentage:.2%})")

2026/08/05 11:47:23 INFO mlflow.tracking.fluent: Experiment with name 'single_car_compare' does not exist. Creating a new experiment.



Запуск эксперимента единичной машины для: 1_raw_dataset...


C:\Users\Степан\AppData\Local\Temp\ipykernel_29284\740521938.py:97: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_single[col] = X_single[col].astype(object).fillna('Unknown').astype(str)
